In [1]:
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers




I0000 00:00:1786599461.026947    8243 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Loading the Dataset

In [2]:
df = pd.read_csv("SMSSpamCollection.csv", encoding = 'latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


### Cleaning the DataSet and Label Encoding

In [3]:
df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
df = df.rename(columns={'v1': 'label', 'v2': 'Text'})
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,Text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


### Split Data and convert to NumPy arrays

In [4]:
X_train,  X_test, y_train, y_test = train_test_split(
    df['Text'],
    df['label_enc'],
    test_size=0.2,
    random_state=42
)

X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()
y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()



### Compute text Statistics for Vectorization

In [5]:
avg_words_per_message = round(df['Text'].str.split().str.len().mean())
total_unique_words = len(set(" ".join(df['Text']).split()))

print(f"Data Loaded. Training samples: {len(X_train_np)}")
print(f"Average words per message: {avg_words_per_message}")
print(f"Approximate vocabulary size: {total_unique_words}")


Data Loaded. Training samples: 4457
Average words per message: 15
Approximate vocabulary size: 15583


### Helper Functions for training and evaluation

In [16]:
def compile_and_fit(model, epochs =5):
    model.compile(
        optimizer = 'adam',
        loss = 'binary_crossentropy', 
        metrics = ['accuracy']
    )
    history = model.fit(
    X_train_np,
    y_train_np,
    epochs = epochs,
    validation_data=(X_test_np, y_test_np)
    )

    return history

def get_metrics(model, X, y):
    y_preds = np.round(model.predict(X))
    return {
        'accuracy': accuracy_score(y, y_preds), 
        'precision': precision_score(y, y_preds),
        'recall': recall_score(y, y_preds),
        "f1-score": f1_score(y, y_preds)
    }


### TextVectorization Layer

In [7]:
from tensorflow.keras.layers import TextVectorization

In [8]:
text_vec = TextVectorization(
    max_tokens = total_unique_words,
    standardize = 'lower_and_strip_punctuation',
    output_sequence_length = avg_words_per_message
)

text_vec.adapt(X_train_np)

E0000 00:00:1786599513.174929    8243 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


### Model 1 - Dense embedding model (build and train)

In [17]:
input_layer = layers.Input(shape=(1,), dtype=tf.string)
x = text_vec(input_layer)
x = layers.Embedding(input_dim = total_unique_words, output_dim= 128)(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(32, activation= 'relu')(x)
output_layer =  layers.Dense(1, activation = 'sigmoid')(x)

model_1 = keras.Model(input_layer, output_layer, name = "Dense_Model")
history_1 = compile_and_fit(model_1)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.9044 - loss: 0.2671 - val_accuracy: 0.9632 - val_loss: 0.1539
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.9812 - loss: 0.0789 - val_accuracy: 0.9749 - val_loss: 0.0877
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.9899 - loss: 0.0333 - val_accuracy: 0.9812 - val_loss: 0.0840
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.9953 - loss: 0.0181 - val_accuracy: 0.9794 - val_loss: 0.0797
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.9984 - loss: 0.0098 - val_accuracy: 0.9794 - val_loss: 0.0843
